# tabular_data_prep - raw OMOP tables -> a modelling-ready tabular matrix + a global selected feature set (v2)
Revised pipeline for the full-record baselines **and** for the RF / XGBoost landmark models, which read the outputs of this notebook.

**What changed from `tabular_ehr_baselines`**
- **Lab windows:** 8-week (56-day) windows, 4-week stride, `[w*7, w*7 + 56)` days from episode start, per-window **min and max**.
- **Missingness is not conflated with absence.** Only PRESENT (patient, feature) entries are stored. Presence is the sparse structure, so a first occurrence in
  week 0 (an explicit stored 0) is "present", absent is "not stored". No sentinel, no stored `_missing` flags, no `(X != 0)`.
  At model time: presence column (0/1) + value column (mean-imputed, i.e. 'value + indicator').
- **Split is read, not regenerated:** from `protopregnet_landmark_cohort` (the ProtoPregNet 8-1-1 split). Every patient in the original ProtoPregNet export's test set is
  asserted to be in the split's test set; the split may hold extra test patients the export did not evaluate, and their count is reported. Every selection statistic below uses TRAIN rows only; validation is left for hyperparameter tuning.
- **Feature selection funnel (train only):** prevalence floor -> [FDR on presence + imputed values] AND [chi-square on presence + odds-ratio floor + BH -> L1 logistic, 1-SE rule].
  The presence screen alone would block labs whose RESULT separates the labels but whose presence does not, so by default (`screen_mode="presence_or_value"`) lab windows whose min/max
  columns FDR selected are also admitted to the L1 step. Event first-occurrence timing is not rescued.
- `episode_end` = `greatest(inferred, recorded)`; labels as before. `*_cohort` tables are used as given: no date cuts, no concept rollup, labs already harmonized.

**Stages (`stage` widget):** `features` (Spark: long tables) -> `matrix` (sparse matrix in Delta) -> `select` (funnel; writes `selected_features`). `all` runs them in order.
Each stage skips work whose output already exists unless `force=true`.

**Outputs:** `tabular_v2_events_long`, `tabular_v2_measurement_long`, `tabular_v2_sparse_long` (+ person / feature index), `tabular_v2_selected_features`
(every candidate design column with its per-stage flags, concept names, `selected_final`), `tabular_v2_selection_stage_counts`, and `meta.json` (window parameters,
matrix fingerprint) that the landmark notebooks read so nothing can silently mismatch.

In [0]:
import os, time, json, gc, shutil, pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType, DoubleType, BooleanType)

dbutils.widgets.dropdown("stage", "all", ["all", "features", "matrix", "select"])
dbutils.widgets.text("n_jobs", "8")
dbutils.widgets.dropdown("force", "false", ["false", "true"])
STAGE = dbutils.widgets.get("stage")
N_JOBS = int(dbutils.widgets.get("n_jobs"))
FORCE = dbutils.widgets.get("force") == "true"

PERSON_ID_COL = "person_id"
CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
LABEL_COLS = ["label_control", "label_pe_ghtn", "label_gdm"]          # defined from the cohort table, below
SPLIT_LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]                 # the same labels as stored in the landmark cohort table

CFG = {
    # ---- sources (unchanged from the baselines notebook) ----
    "cohort_table": "catalog.schema.step7_filtered_cohort_with_apos",
    "condition_table": "catalog.schema.condition_occurrence_cohort",
    "procedure_table": "catalog.schema.procedure_occurrence_cohort",
    "drug_table": "catalog.schema.drug_exposure_cohort",
    "measurement_table": "catalog.schema.measurement_cohort",
    "staging_schema": "catalog.schema",
    "cond_cols": ("condition_concept_id", "condition_start_date"), "proc_cols": ("procedure_concept_id", "procedure_date"),
    "drug_cols": ("drug_concept_id", "drug_exposure_start_date"), "meas_cols": ("measurement_concept_id", "measurement_date", "value_as_number"),
    "concept_table": "catalog.omop.concept",                       # (concept_id, concept_name) to label the selected features; skipped if unavailable

    # ---- split: READ from the landmark cohort (the ProtoPregNet 8-1-1 split), never regenerated ----
    "split_table": "catalog.schema.protopregnet_landmark_cohort",
    "original_export_path": "/path/to/exports/full_analysis_export.pkl",   # its pids_test must equal the split's test set exactly

    # ---- time windows ----
    "step_days": 7,               # gestational week = floor(day_offset / 7), day_offset = days from episode_start
    "window_start_wk": -26,       # 6 months before episode start
    "window_end_wk": 42,
    "window_len_days": 56,        # 8-week windows ...
    "stride_weeks": 4,            # ... starting every 4 weeks (50% overlap): window w covers days [w*7, w*7 + 56)
    "concept_batch_size_meas": 15,

    # ---- outputs ----
    "events_long_table": "catalog.schema.tabular_v2_events_long",
    "measurement_long_table": "catalog.schema.tabular_v2_measurement_long",
    "sparse_long_table": "catalog.schema.tabular_v2_sparse_long",
    "person_index_table": "catalog.schema.tabular_v2_person_index",
    "feature_index_table": "catalog.schema.tabular_v2_feature_index",
    "selected_table": "catalog.schema.tabular_v2_selected_features",
    "stage_counts_table": "catalog.schema.tabular_v2_selection_stage_counts",
    "export_dir": "/path/to/exports/tabular_v2",    # meta.json (window parameters + matrix fingerprint) for the landmark notebooks
    "staging_dir": "/path/to/exports/tabular_v2/staging",

    # ---- selection funnel (all statistics on TRAIN rows only) ----
    "min_prevalence": 0.01,       # a group (concept / concept-window) must be present in >= 1% of train patients
    "fdr_alpha": 0.05,            # FDR (ANOVA F, Benjamini-Hochberg) on [presence | mean-imputed values]
    "bh_alpha": 0.05,             # chi-square presence screen: BH level ...
    "or_floor": 1.15,             # ... AND odds ratio >= 1.15 or <= 1/1.15 (significant AND big enough to matter)
    "screen_mode": "presence_or_value",  # default: lab windows whose min/max FDR selected also pass the presence screen, so labs whose RESULT (not their presence) separates
                                  # the labels can reach the final set. "presence" = the presence screen alone; with the AND below it can never select such labs.
    "intersection_scope": "per_label",  # "per_label": a column must pass FDR AND the funnel for the SAME label (then union over labels) | "global": FDR for any label AND funnel for any
    "l1_Cs": [float(c) for c in np.logspace(-4, -0.5, 10)],   # C grid (smaller C = stronger penalty); widen if a 'grid edge' warning appears
    "l1_cv": 5, "l1_max_iter": 200, "l1_tol": 1e-3,
    "seed": 42, "n_jobs": N_JOBS,
    "floors_to_report": [0.001, 0.005, 0.01, 0.02, 0.05],           # counts of groups surviving each floor, for information only
}
print(f"stage={STAGE} | force={FORCE} | n_jobs={N_JOBS} | sklearn {sklearn.__version__} | numpy {np.__version__}")

In [0]:
# ===== shared design code (identical in the RF / XGBoost landmark notebooks) =====
# ============================================================================
# Shared design code (prep + RF + XGBoost notebooks): stored-column parsing, structural censoring, design matrix.
# STORED representation (tabular_v2): one sparse entry per PRESENT (patient, stored column); absent entries are simply not stored.
#   events:  cond_<id> / proc_<id> / drug_<id>   value = first-occurrence gestational week (can be 0 or negative: an explicit stored value)
#   labs:    meas_<id>_wk<w>_min / _max          min and max of the results in the window [w*7, w*7 + window_len) days
# PRESENCE is therefore the sparse STRUCTURE. A first occurrence in week 0 is an explicit stored 0 and stays "present". Never use
# .toarray(), `!= 0` or eliminate_zeros() on this matrix to decide presence.
# ============================================================================
STORED_PATTERN = r"^(?:(?P<ev_kind>cond|proc|drug)_(?P<ev_id>-?\d+)|meas_(?P<lab_id>-?\d+)_wk(?P<wk>-?\d+)_(?P<stat>min|max))$"


def parse_stored_columns(feature_names):
    """One row per stored column: kind (event | lab | unparsed), group (the concept for events, the concept-window for labs), stat, window_wk."""
    s = pd.Series(list(feature_names), name="col")
    ex = s.str.extract(STORED_PATTERN)
    kind = np.where(ex["ev_kind"].notna(), "event", np.where(ex["lab_id"].notna(), "lab", "unparsed"))
    group = np.where(kind == "lab", s.str.replace(r"_(min|max)$", "", regex=True), s)
    out = pd.DataFrame({"col": s, "kind": kind, "group": group,
                        "stat": np.where(kind == "event", "week", ex["stat"].fillna("")),
                        "concept_id": pd.to_numeric(ex["ev_id"].fillna(ex["lab_id"]), errors="coerce").astype("Int64"),
                        "window_wk": pd.to_numeric(ex["wk"], errors="coerce").astype("Int64")})
    return out


def censor_struct(Xrows, k, info, step_days, window_len_days):
    """Keep only what is known at the landmark (cutoff = episode_start + 7k days; events ON the cutoff day are not used) by REMOVING entries
    from the sparse structure (not zeroing them), so a first occurrence in week 0 that is still visible stays 'present'.
      events: kept iff first-occurrence week <= k-1 (a later first occurrence is, at the cutoff, absent)
      labs:   kept iff the window's last day <= day 7k (a window spanning the cutoff would contain future results); min and max go together.
    Xrows: CSR rows (a private copy is NOT required: a new matrix is returned)."""
    is_event = (info["kind"] == "event").to_numpy()
    is_lab = (info["kind"] == "lab").to_numpy()
    last_day = np.where(is_lab, info["window_wk"].astype("float").to_numpy() * step_days + window_len_days - 1, np.inf)
    lab_ok = is_lab & (last_day <= step_days * k)
    j = Xrows.indices
    keep = np.where(is_event[j], Xrows.data <= k - 1, lab_ok[j])
    n = Xrows.shape[0]
    row_of_entry = np.repeat(np.arange(n), np.diff(Xrows.indptr))
    indptr = np.concatenate([[0], np.cumsum(np.bincount(row_of_entry[keep], minlength=n))])
    return sp.csr_matrix((Xrows.data[keep], Xrows.indices[keep], indptr), shape=Xrows.shape)


def make_design_dense(Xc, base_idx, is_presence, impute):
    """Dense float32 design for the SELECTED design columns, in the order given.
      presence column: 1 if the stored base column has an entry (structure), else 0
      value column:    the stored value if present, else impute (train mean of the present values)
    Mean-imputation + the presence column is the standard 'value + indicator' encoding; trees are invariant to the constant, so no scaling or
    centering is applied here."""
    n = Xc.shape[0]
    sub = Xc[:, np.asarray(base_idx)].tocoo()
    out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
    out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
    return out


def matrix_fingerprint(feature_names, shape):
    """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
    import hashlib
    h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
    return h[:16]

In [0]:
# ===== Spark feature construction =====
# ============================================================================
# Spark feature construction (prep notebook). Everything is written directly in LONG format (person_id, feature_name, value), only for PRESENT
# entries: no wide pivots, no sentinel, no flag columns. Absence is simply the absence of a row.
# ============================================================================
def window_grid_specs(start_wk, end_wk, stride_wk, len_days, step_days):
    """(window_wk, start_day, end_day) with end exclusive: a result on day d belongs to window w iff start_day <= d < end_day."""
    return [(int(wk), int(wk * step_days), int(wk * step_days + len_days)) for wk in range(start_wk, end_wk + 1, stride_wk)]


def event_features_long(events_df, concept_col, date_col, prefix, cohort_df, person_col, step_days):
    """First occurrence of each concept per patient -> gestational week = floor(days from episode_start / 7). Can be 0 or negative (explicit values).
    Rows with a NULL concept are dropped (they cannot form a feature name). No date cuts: the *_cohort tables are used as given."""
    first_occ = (events_df.select(person_col, concept_col, date_col)
                 .where(F.col(concept_col).isNotNull() & ~F.isnan(F.col(concept_col).cast("double")))      # NULL / NaN cannot form a feature name (concept 0 is kept)
                 .groupBy(person_col, concept_col).agg(F.min(date_col).alias("first_date"))
                 .join(cohort_df.select(person_col, "episode_start"), on=person_col, how="inner")
                 .withColumn("day_offset", F.datediff(F.col("first_date"), F.col("episode_start"))))
    return first_occ.selectExpr(person_col, f"concat('{prefix}_', CAST(CAST({concept_col} AS BIGINT) AS STRING)) AS feature_name",     # BIGINT first: a double concept id would give 'cond_101.0'
                                f"CAST(floor(day_offset / {int(step_days)}) AS FLOAT) AS value")


def add_day_offset(meas_df, cohort_df, person_col, concept_col, date_col, value_col):
    v = F.col(value_col).cast("double")
    return (meas_df.select(person_col, concept_col, date_col, v.alias(value_col))
            .where(v.isNotNull() & ~F.isnan(v))                                    # NULL and NaN results cannot be summarised (non-numeric results were handled upstream)
            .join(cohort_df.select(person_col, "episode_start"), on=person_col, how="inner")
            .withColumn("day_offset", F.datediff(F.col(date_col), F.col("episode_start"))))


def lab_window_features_long(meas_offset_df, window_grid_df, person_col, concept_col, value_col):
    """Per (patient, concept, window): min and max of the results. Overlapping windows: a result can fall in two windows."""
    windowed = (meas_offset_df.crossJoin(F.broadcast(window_grid_df))
                .where((F.col("day_offset") >= F.col("win_start_day")) & (F.col("day_offset") < F.col("win_end_day"))))
    stats = (windowed.groupBy(person_col, concept_col, "window_wk")
             .agg(F.min(value_col).alias("v_min"), F.max(value_col).alias("v_max")))
    name = lambda stat: f"concat('meas_', CAST(CAST({concept_col} AS BIGINT) AS STRING), '_wk', window_wk, '_{stat}')"
    return (stats.selectExpr(person_col, f"{name('min')} AS feature_name", "CAST(v_min AS FLOAT) AS value")
            .unionByName(stats.selectExpr(person_col, f"{name('max')} AS feature_name", "CAST(v_max AS FLOAT) AS value")))

In [0]:
# ===== selection funnel =====
# ============================================================================
# Feature-selection funnel (prep notebook). Every statistic is computed on TRAIN rows only.
#   0  prevalence floor on groups (a concept, or a concept-window): present in >= min_prevalence of train patients
#   F  FDR (ANOVA F + Benjamini-Hochberg) on the design [presence | mean-imputed values]          -> fdr set, per label
#   1  chi-square on PRESENCE + minimum odds-ratio effect size + BH                               -> groups that pass, per label
#      (screen_mode "presence_or_value", the default, also lets through LAB windows whose min/max columns FDR selected: labs whose RESULT
#       separates the labels but whose presence does not. Event first-occurrence timing is not rescued. "presence" = the presence screen alone)
#   2  L1 (lasso) logistic regression on the design columns of the groups that pass, C by 5-fold CV with the 1-SE rule -> funnel set, per label
#   final = FDR AND funnel (intersection), per label then unioned across labels (or globally: see CFG intersection_scope)
# ============================================================================
def build_candidate_spec(X_train, info, min_prevalence):
    """Candidate design columns after the prevalence floor.
    spec: design_col, role (presence | value), group, kind, stat, base_idx (stored column), base_col, impute_value.
    Presence column: 1 if the group's anchor stored column has an entry (events: the week column; labs: the _min column; min and max are always stored together).
    Value column: the stored value; absent -> impute_value = train mean of the PRESENT values."""
    n_train, n_cols = X_train.shape
    present_n = np.bincount(X_train.indices, minlength=n_cols)                       # stored entries per column (explicit zeros count as present)
    sum_v = np.bincount(X_train.indices, weights=X_train.data.astype("float64"), minlength=n_cols)
    mean_present = np.divide(sum_v, present_n, out=np.zeros(n_cols), where=present_n > 0)
    inf = info.copy(); inf["idx"] = np.arange(n_cols); inf["present_n"] = present_n; inf["mean_present"] = mean_present
    inf = inf[inf["kind"] != "unparsed"]
    anchor = (inf.assign(rank=inf["stat"].map({"week": 0, "min": 0, "max": 1}))
                 .sort_values(["group", "rank", "idx"]).groupby("group", sort=False).head(1))
    groups = anchor[["group", "kind", "idx", "present_n"]].rename(columns={"idx": "anchor_idx"}).reset_index(drop=True)
    groups["prevalence"] = groups["present_n"] / n_train
    groups["kept"] = groups["prevalence"] >= min_prevalence
    kept = groups[groups["kept"]]
    pres = pd.DataFrame({"design_col": "has::" + kept["group"], "role": "presence", "group": kept["group"], "kind": kept["kind"], "stat": "presence",
                         "base_idx": kept["anchor_idx"], "base_col": [info["col"].iloc[i] for i in kept["anchor_idx"]], "impute_value": 0.0})
    val = inf[inf["group"].isin(kept["group"])]
    vals = pd.DataFrame({"design_col": val["col"], "role": "value", "group": val["group"], "kind": val["kind"], "stat": val["stat"],
                         "base_idx": val["idx"], "base_col": val["col"], "impute_value": val["mean_present"]})
    spec = pd.concat([pres, vals], ignore_index=True)
    spec["base_idx"] = spec["base_idx"].astype(int)
    return spec, groups


def sparse_design(Xrows, spec):
    """Sparse design for the linear steps: presence = 1 (structure), value = (value - impute). Because absent values equal the imputed mean, absent
    becomes 0 after centering, so mean-imputation costs no density. Tests (F, L1) are invariant to the shift."""
    sub = Xrows[:, spec["base_idx"].to_numpy()].tocsr()
    is_p = (spec["role"] == "presence").to_numpy(); imp = spec["impute_value"].to_numpy()
    d = sub.indices
    sub.data = np.where(is_p[d], 1.0, sub.data.astype("float64") - imp[d])
    return sub


def column_scale(Xd):
    """1 / std per column (std around the column mean), for the L1 penalty to treat columns evenly; applied without centering to keep sparsity."""
    m = np.asarray(Xd.mean(0)).ravel(); ex2 = np.asarray(Xd.multiply(Xd).mean(0)).ravel()
    std = np.sqrt(np.maximum(ex2 - m ** 2, 0))
    return np.where(std > 1e-12, 1.0 / np.where(std > 1e-12, std, 1.0), 0.0)


def bh_qvalues(p):
    """Benjamini-Hochberg q-values; NaN p-values stay NaN."""
    p = np.asarray(p, dtype=float); q = np.full(len(p), np.nan)
    ok = np.flatnonzero(~np.isnan(p)); order = ok[np.argsort(p[ok])]
    ranked = p[order] * len(ok) / (np.arange(len(ok)) + 1)
    q[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    return q


def chi2_presence_screen(P, y, or_floor, alpha):
    """Pearson chi-square (1 df, no continuity correction) of presence vs the label for every presence column of P (binary CSR, train rows),
    odds ratio with a +0.5 smoothing, BH across the columns tested. keep = q <= alpha AND the odds ratio is >= or_floor or <= 1/or_floor."""
    from scipy.stats import chi2 as chi2_dist
    N = P.shape[0]; y = np.asarray(y, dtype="float64")
    m1 = np.asarray(P.sum(0)).ravel().astype("float64"); a = np.asarray(P.T @ y).ravel(); P1 = y.sum()
    b, c, d = m1 - a, P1 - a, N - m1 - P1 + a
    num = N * (a * d - b * c) ** 2; den = (a + b) * (c + d) * (a + c) * (b + d)
    stat = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
    p = chi2_dist.sf(stat, 1)
    odds = ((a + .5) * (d + .5)) / ((b + .5) * (c + .5))
    q = bh_qvalues(p)
    keep = (q <= alpha) & ((odds >= or_floor) | (odds <= 1.0 / or_floor))
    return keep, {"chi2": stat, "p": p, "q": q, "odds_ratio": odds}


def fdr_select(Xd, y, alpha):
    from sklearn.feature_selection import SelectFdr, f_classif
    return SelectFdr(f_classif, alpha=alpha).fit(Xd, y).get_support(indices=True)


def l1_one_se(Xs, y, Cs, cv, seed, n_jobs, max_iter, tol):
    """L1 (lasso) logistic regression. C (inverse penalty strength) is chosen by stratified k-fold CV on AUROC with the 1-SE rule: the STRONGEST
    penalty (smallest C) whose mean CV AUROC is within one standard error of the best. liblinear is the fast L1 solver for sparse data
    (it also lightly penalises the intercept, which is harmless here)."""
    from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    folds = StratifiedKFold(cv, shuffle=True, random_state=seed)
    m = LogisticRegressionCV(Cs=list(Cs), cv=folds, penalty="l1", solver="liblinear", scoring="roc_auc", refit=False,
                             n_jobs=n_jobs, max_iter=max_iter, tol=tol, random_state=seed).fit(Xs, y)
    S = m.scores_[1]; Cgrid = np.asarray(m.Cs_)
    mean, se = S.mean(0), S.std(0, ddof=1) / np.sqrt(S.shape[0])
    nz = (np.abs(m.coefs_paths_[1][:, :, :-1]) > 0).sum(2).mean(0)
    order = np.argsort(Cgrid)                                                      # strongest penalty first
    best = int(mean.argmax()); thr = mean[best] - se[best]
    pick = int(next(i for i in order if mean[i] >= thr))
    final = LogisticRegression(penalty="l1", solver="liblinear", C=float(Cgrid[pick]), max_iter=max_iter, tol=tol, random_state=seed).fit(Xs, y)
    table = pd.DataFrame({"C": Cgrid[order], "cv_auroc": mean[order], "se": se[order], "mean_nonzero": nz[order]})
    info = {"best_C": float(Cgrid[best]), "chosen_C": float(Cgrid[pick]), "best_auroc": float(mean[best]), "chosen_auroc": float(mean[pick]),
            "best_at_grid_edge": bool(best in (order[0], order[-1]))}
    return np.flatnonzero(np.abs(final.coef_[0]) > 0), table, info


def run_selection(Xt, Yt, spec, cfg, class_names, log=print):
    """Xt: stored train rows (CSR); Yt: (n_train, n_labels). Returns per-label boolean flags over spec rows plus diagnostics."""
    from scipy import sparse as _sp
    D = sparse_design(Xt, spec)
    pres_pos = np.flatnonzero((spec["role"] == "presence").to_numpy())
    res = {"fdr": {}, "screen_group": {}, "funnel": {}, "final": {}, "cv_tables": {}, "cv_info": {}, "screen_stats": {}}
    n_spec = len(spec)
    for c, name in enumerate(class_names):
        y = Yt[:, c]
        fdr_idx = fdr_select(D, y, cfg["fdr_alpha"])
        keep_pres, st = chi2_presence_screen(D[:, pres_pos], y, cfg["or_floor"], cfg["bh_alpha"])
        pass_groups = set(spec["group"].iloc[pres_pos[keep_pres]])
        is_val = (spec["role"] == "value").to_numpy(); is_lab = (spec["kind"] == "lab").to_numpy()
        fdr_flag = np.isin(np.arange(n_spec), fdr_idx)
        blocked_lab = set(spec["group"][fdr_flag & is_val & is_lab]) - pass_groups       # lab windows FDR likes for their RESULTS but whose presence failed the screen
        blocked_event = set(spec["group"][fdr_flag & is_val & ~is_lab]) - pass_groups     # event concepts FDR likes for their TIMING but whose presence failed
        if blocked_lab:
            log(f"  NOTE {name}: {len(blocked_lab):,} lab windows have FDR-selected result columns but fail the presence screen -> "
                + ("admitted to L1 (screen_mode=presence_or_value)" if cfg["screen_mode"] == "presence_or_value" else "they cannot reach the final set (screen_mode=presence)"))
        if blocked_event:
            log(f"  NOTE {name}: {len(blocked_event):,} event concepts have an FDR-selected first-occurrence week but fail the presence screen; event timing is not rescued")
        if cfg["screen_mode"] == "presence_or_value":
            pass_groups |= blocked_lab
        cand = np.flatnonzero(spec["group"].isin(pass_groups).to_numpy())
        if len(cand):
            Dc = D[:, cand]
            Dc = Dc @ _sp.diags(column_scale(Dc))
            sel_local, table, info = l1_one_se(Dc.tocsr(), y, cfg["l1_Cs"], cfg["l1_cv"], cfg["seed"], cfg["n_jobs"], cfg["l1_max_iter"], cfg["l1_tol"])
            funnel_idx = cand[sel_local]
            res["cv_tables"][name], res["cv_info"][name] = table, info
            if info["best_at_grid_edge"]:
                log(f"  WARNING {name}: the best CV score is at the edge of the C grid; widen CFG['l1_Cs']")
        else:
            funnel_idx = np.array([], dtype=int)
        flag = lambda idx: np.isin(np.arange(n_spec), idx)
        res["fdr"][name], res["funnel"][name] = flag(fdr_idx), flag(funnel_idx)
        res["screen_group"][name] = spec["group"].isin(pass_groups).to_numpy()
        res.setdefault("blocked_lab_groups", {})[name] = len(blocked_lab)
        res.setdefault("blocked_event_groups", {})[name] = len(blocked_event)
        res["final"][name] = res["fdr"][name] & res["funnel"][name]
        res["screen_stats"][name] = st
        log(f"  {name}: FDR {len(fdr_idx):,} | presence screen keeps {len(pass_groups):,} groups -> L1 on {len(cand):,} columns keeps {len(funnel_idx):,} "
            f"| FDR AND funnel {int(res['final'][name].sum()):,}")
    any_ = lambda d: np.any(np.column_stack([d[n] for n in class_names]), axis=1)
    if cfg["intersection_scope"] == "per_label":
        res["selected_final"] = any_(res["final"])                                 # passes BOTH methods for at least one label
    else:
        res["selected_final"] = any_(res["fdr"]) & any_(res["funnel"])             # passes FDR for some label AND the funnel for some label
    return res

In [0]:
# ===== sparse-matrix assembly / Delta persistence: copied unchanged from tabular_ehr_baselines =====
FULL_SPARSE_LONG_TABLE = CFG['sparse_long_table']
PERSON_INDEX_TABLE = CFG['person_index_table']
FEATURE_INDEX_TABLE = CFG['feature_index_table']

def _chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def build_sparse_feature_matrix(long_table_names, person_index, pull_chunk_size=500):
    person_id_to_row = {pid: i for i, pid in enumerate(person_index)}
    feat_name_to_col = {}

    table_counts = {}
    for table_name in long_table_names:
        print(f"  Pass 1 (counting) {table_name}...")
        total = 0
        for i in range(0, len(person_index), pull_chunk_size):
            batch_pids = person_index[i:i + pull_chunk_size]
            id_list_sql = ", ".join(f"'{pid}'" for pid in batch_pids)
            cnt_df = spark.sql(
                f"SELECT feature_name, COUNT(*) as n FROM {table_name} "
                f"WHERE {PERSON_ID_COL} IN ({id_list_sql}) GROUP BY feature_name"
            ).toPandas()
            for feat in cnt_df["feature_name"]:
                if feat not in feat_name_to_col:
                    feat_name_to_col[feat] = len(feat_name_to_col)
            total += int(cnt_df["n"].sum())
        table_counts[table_name] = total
        print(f"    {total:,} entries")

    total_nnz = sum(table_counts.values())
    n_cols = len(feat_name_to_col)
    print(f"  Total: {total_nnz:,} entries, {n_cols:,} features -- pre-allocating buffers")

    rows_arr = np.empty(total_nnz, dtype=np.int32)
    cols_arr = np.empty(total_nnz, dtype=np.int32)
    vals_arr = np.empty(total_nnz, dtype=np.float32)
    pos = 0

    for table_name in long_table_names:
        print(f"  Pass 2 (loading) {table_name}...")
        for i in range(0, len(person_index), pull_chunk_size):
            batch_pids = person_index[i:i + pull_chunk_size]
            id_list_sql = ", ".join(f"'{pid}'" for pid in batch_pids)
            query = f"SELECT {PERSON_ID_COL}, feature_name, value FROM {table_name} WHERE {PERSON_ID_COL} IN ({id_list_sql})"
            chunk_df = spark.sql(query).toPandas()
            n = len(chunk_df)
            if n > 0:
                rows_arr[pos:pos+n] = chunk_df[PERSON_ID_COL].map(person_id_to_row).values
                cols_arr[pos:pos+n] = chunk_df["feature_name"].map(feat_name_to_col).values
                vals_arr[pos:pos+n] = chunk_df["value"].values.astype("float32")
                pos += n
            del chunk_df
            gc.collect()

    assert pos == total_nnz, f"count mismatch: filled {pos}, expected {total_nnz}"

    n_rows = len(person_index)
    X_sparse = sp.csr_matrix((vals_arr, (rows_arr, cols_arr)), shape=(n_rows, n_cols))
    feature_names = [None] * n_cols
    for name, idx in feat_name_to_col.items():
        feature_names[idx] = name

    density = X_sparse.nnz / (n_rows * n_cols) * 100
    mem_mb = (X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes) / 1e6
    print(f"\n  Sparse matrix: {n_rows:,} x {n_cols:,}, {X_sparse.nnz:,} nonzero "
          f"({density:.3f}% dense), {mem_mb:.1f} MB")
    return X_sparse, feature_names


def save_sparse_matrix_to_delta(person_index, feature_names, rows_arr, cols_arr, vals_arr,
                                write_chunk_size=5_000_000):
    """Persists a built sparse matrix's (row_idx, col_idx, value) triplets
    to Delta, PLUS two small metadata tables mapping row_idx<->person_id
    and col_idx<->feature_name. Reload never needs to re-derive these
    mappings via an expensive DISTINCT scan over the full (here, ~681M
    row) combined table -- they're saved once, directly. Verified this
    save/load design round-trips to an EXACTLY identical sparse matrix
    (same values, same row order, same column order) before using it."""
    n_rows, n_cols, total_nnz = len(person_index), len(feature_names), len(rows_arr)
    print(f"Saving sparse matrix ({n_rows:,} x {n_cols:,}, {total_nnz:,} nonzero) to Delta...")

    # small metadata tables first -- cheap, needed for any future reload
    person_meta = pd.DataFrame({"person_id": list(person_index), "row_idx": range(n_rows)})
    spark.createDataFrame(person_meta).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(PERSON_INDEX_TABLE)
    print(f"  Saved {PERSON_INDEX_TABLE} ({n_rows:,} rows)")

    feature_meta = pd.DataFrame({"feature_name": feature_names, "col_idx": range(n_cols)})
    spark.createDataFrame(feature_meta).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(FEATURE_INDEX_TABLE)
    print(f"  Saved {FEATURE_INDEX_TABLE} ({n_cols:,} rows)")

    # big combined long table, WRITTEN IN CHUNKS -- same OOM-avoidance
    # principle as everywhere else in this pipeline: never materialize
    # more than one chunk's worth of pandas/Arrow data at a time
    n_chunks = (total_nnz + write_chunk_size - 1) // write_chunk_size
    for i, start in enumerate(range(0, total_nnz, write_chunk_size)):
        end = min(start + write_chunk_size, total_nnz)
        chunk_df = pd.DataFrame({
            "row_idx": rows_arr[start:end],
            "col_idx": cols_arr[start:end],
            "value": vals_arr[start:end],
        })
        mode = "overwrite" if i == 0 else "append"
        spark.createDataFrame(chunk_df).write.format("delta").mode(mode) \
            .option("overwriteSchema", str(i == 0).lower()).saveAsTable(FULL_SPARSE_LONG_TABLE)
        print(f"  chunk {i+1}/{n_chunks} written ({end-start:,} rows)")
        del chunk_df
        gc.collect()

    print(f"Done. Saved to {FULL_SPARSE_LONG_TABLE}.")


def load_sparse_matrix_from_delta(persons_per_chunk=500):
    """Chunks strictly by PATIENT COUNT (row_idx range spanning
    `persons_per_chunk` patients at a time) -- not derived from any
    nnz-sized number. The previous version divided n_rows (patient
    count, ~59,826) by an nnz-sized pull_chunk_size (20,000,000),
    which silently collapsed to n_chunks=1 -- an unbounded single-shot
    load of all 681M rows that killed the Spark driver. This version
    can't degenerate that way: persons_per_chunk directly controls
    chunk count, with no division that can round down to 1.
    """
    person_meta = spark.table(PERSON_INDEX_TABLE).toPandas().sort_values("row_idx")
    feature_meta = spark.table(FEATURE_INDEX_TABLE).toPandas().sort_values("col_idx")
    person_index = person_meta["person_id"].tolist()
    feature_names = feature_meta["feature_name"].tolist()
    n_rows, n_cols = len(person_index), len(feature_names)

    total_nnz = spark.sql(f"SELECT COUNT(*) as n FROM {FULL_SPARSE_LONG_TABLE}").collect()[0]["n"]
    print(f"Loading {n_rows:,} x {n_cols:,} matrix, {total_nnz:,} nonzero...")

    rows_arr = np.empty(total_nnz, dtype=np.int32)
    cols_arr = np.empty(total_nnz, dtype=np.int32)
    vals_arr = np.empty(total_nnz, dtype=np.float32)
    pos = 0

    chunk_starts = list(range(0, n_rows, persons_per_chunk))
    n_chunks = len(chunk_starts)
    assert n_chunks >= 2, f"only {n_chunks} chunk(s) computed for {n_rows:,} patients -- refusing to run unchunked"
    print(f"  {n_chunks} chunks of ~{persons_per_chunk} patients each")

    for i, start in enumerate(chunk_starts):
        end = min(start + persons_per_chunk, n_rows)
        chunk_df = spark.sql(
            f"SELECT row_idx, col_idx, value FROM {FULL_SPARSE_LONG_TABLE} "
            f"WHERE row_idx >= {start} AND row_idx < {end}"
        ).toPandas()
        n = len(chunk_df)
        rows_arr[pos:pos+n] = chunk_df["row_idx"].values
        cols_arr[pos:pos+n] = chunk_df["col_idx"].values
        vals_arr[pos:pos+n] = chunk_df["value"].values
        pos += n
        print(f"  chunk {i+1}/{n_chunks} loaded ({n:,} rows, patients [{start}, {end}))")
        del chunk_df
        gc.collect()

    if pos != total_nnz:
        print(f"  NOTE: filled {pos:,} of {total_nnz:,} allocated -- trimming buffers.")
        rows_arr, cols_arr, vals_arr = rows_arr[:pos], cols_arr[:pos], vals_arr[:pos]

    X_sparse = sp.csr_matrix((vals_arr, (rows_arr, cols_arr)), shape=(n_rows, n_cols))
    print(f"Loaded: {X_sparse.shape}, {X_sparse.nnz:,} nonzero.")
    return X_sparse, feature_names, pd.Index(person_index, name=PERSON_ID_COL)

In [0]:
# ============================================================================
# cohort (episode_end = greatest(inferred, recorded), as requested) and person index
# ============================================================================
cohort = (spark.table(CFG["cohort_table"])
          .withColumn("episode_start", F.least(F.col("inferred_episode_start"), F.col("recorded_episode_start")))
          .withColumn("episode_end", F.greatest(F.col("inferred_episode_end"), F.col("recorded_episode_end")))
          .withColumn("label_control", F.col("is_control").cast("int"))
          .withColumn("label_gdm", F.col("has_gdm").cast("int"))
          .withColumn("label_pe_ghtn", F.when((F.col("has_ghtn").cast("boolean") == True) | (F.col("has_pe").cast("boolean") == True)
                                              | (F.col("has_inferred_pe").cast("boolean") == True), F.lit(1)).otherwise(F.lit(0)))
          .select(PERSON_ID_COL, "episode_start", "episode_end", *LABEL_COLS))
all_person_ids = sorted(r[0] for r in cohort.select(PERSON_ID_COL).distinct().collect())
full_person_index = pd.Index(all_person_ids, name=PERSON_ID_COL)
print(f"{len(full_person_index):,} patients in the cohort")

def publish_text(text, volume_path):
    """Small files: local disk first, then a sequential copy (Volumes reject non-sequential writes)."""
    os.makedirs("/tmp/tabular_v2_publish", exist_ok=True)
    local = f"/tmp/tabular_v2_publish/{os.path.basename(volume_path)}"
    with open(local, "w") as f:
        f.write(text)
    os.makedirs(os.path.dirname(volume_path), exist_ok=True)
    try:
        dbutils.fs.cp(f"file:{local}", volume_path)
    except Exception:
        shutil.copyfile(local, volume_path)

def table_exists(name):
    return spark.catalog.tableExists(name)

In [0]:
# ============================================================================
# stage "features": long tables of PRESENT entries only
#   events: first-occurrence gestational week per concept (cond_/proc_/drug_<id>)
#   labs:   per concept x window: min and max (meas_<id>_wk<w>_min / _max)
# ============================================================================
if STAGE in ("all", "features"):
    window_specs = window_grid_specs(CFG["window_start_wk"], CFG["window_end_wk"], CFG["stride_weeks"], CFG["window_len_days"], CFG["step_days"])
    window_grid = spark.createDataFrame(window_specs, ["window_wk", "win_start_day", "win_end_day"])
    print(f"{len(window_specs)} windows of {CFG['window_len_days']} days every {CFG['stride_weeks']} weeks, e.g. {window_specs[:2]} ... {window_specs[-1]}")

    if FORCE or not table_exists(CFG["events_long_table"]):
        parts = []
        for prefix, tbl, (cc, dc) in [("cond", CFG["condition_table"], CFG["cond_cols"]), ("proc", CFG["procedure_table"], CFG["proc_cols"]),
                                      ("drug", CFG["drug_table"], CFG["drug_cols"])]:
            parts.append(event_features_long(spark.table(tbl), cc, dc, prefix, cohort, PERSON_ID_COL, CFG["step_days"]))
        events_long = parts[0].unionByName(parts[1]).unionByName(parts[2])
        events_long.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CFG["events_long_table"])
        print(f"events written: {spark.table(CFG['events_long_table']).count():,} rows -> {CFG['events_long_table']}")
    else:
        print(f"{CFG['events_long_table']} exists (force=true rebuilds)")

    if FORCE or not table_exists(CFG["measurement_long_table"]):
        cc, dc, vc = CFG["meas_cols"]
        staging_table = f"{CFG['staging_schema']}.tmp_tabular_v2_meas_offset"
        # materialise day offsets once (partitioned by concept) so each concept batch below reads it instead of recomputing the join
        (add_day_offset(spark.table(CFG["measurement_table"]), cohort, PERSON_ID_COL, cc, dc, vc)
         .write.mode("overwrite").partitionBy(cc).saveAsTable(staging_table))
        meas_off = spark.table(staging_table)
        concept_ids = [r[0] for r in meas_off.select(cc).distinct().collect()]
        n_batches = (len(concept_ids) + CFG["concept_batch_size_meas"] - 1) // CFG["concept_batch_size_meas"]
        print(f"measurements: {len(concept_ids):,} concepts in {n_batches} batches -> {CFG['measurement_long_table']}")
        for i, batch in enumerate(_chunked(concept_ids, CFG["concept_batch_size_meas"])):
            long_b = lab_window_features_long(meas_off.where(F.col(cc).isin(batch)), window_grid, PERSON_ID_COL, cc, vc)
            (long_b.write.format("delta").mode("overwrite" if i == 0 else "append")
             .option("overwriteSchema", str(i == 0).lower()).saveAsTable(CFG["measurement_long_table"]))
            print(f"  batch {i + 1}/{n_batches} written")
        spark.sql(f"DROP TABLE IF EXISTS {staging_table}")
    else:
        print(f"{CFG['measurement_long_table']} exists (force=true rebuilds)")

In [0]:
# ============================================================================
# stage "matrix": sparse CSR (patients x stored columns) from the long tables; saved to Delta
# ============================================================================
def check_stored_matrix(X, feature_names):
    info = parse_stored_columns(feature_names)
    assert (info["kind"] != "unparsed").all(), f"unparsed feature names: {info.loc[info['kind'] == 'unparsed', 'col'].head().tolist()}"
    assert len(set(feature_names)) == len(feature_names), "duplicate feature names"
    return info

if STAGE in ("all", "matrix"):
    if FORCE or not table_exists(CFG["sparse_long_table"]):
        X, feature_names = build_sparse_feature_matrix([CFG["events_long_table"], CFG["measurement_long_table"]], full_person_index)
        info = check_stored_matrix(X, feature_names)
        X_coo = X.tocoo()            # explicit zeros (week-0 first occurrences) are kept: presence is the STRUCTURE
        save_sparse_matrix_to_delta(full_person_index, feature_names, X_coo.row, X_coo.col, X_coo.data)
    else:
        print(f"{CFG['sparse_long_table']} exists (force=true rebuilds): loading")
        X, feature_names, full_person_index = load_sparse_matrix_from_delta()
        info = check_stored_matrix(X, feature_names)
    FINGERPRINT = matrix_fingerprint(feature_names, X.shape)
    print(f"stored matrix {X.shape}, {X.nnz:,} entries ({int((X.data == 0).sum()):,} explicit zeros = week-0 / zero-valued entries that remain PRESENT) | "
          f"columns: {info['kind'].value_counts().to_dict()} | fingerprint {FINGERPRINT}")
    publish_text(json.dumps({"version": "tabular_v2", "fingerprint": FINGERPRINT, "shape": list(X.shape), "step_days": CFG["step_days"],
                             "window_start_wk": CFG["window_start_wk"], "window_end_wk": CFG["window_end_wk"], "window_len_days": CFG["window_len_days"],
                             "stride_weeks": CFG["stride_weeks"],
                             "created": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2), f"{CFG['export_dir']}/meta.json")

In [0]:
# ============================================================================
# stage "select" (1/2): split + labels READ from the landmark cohort and verified
# ============================================================================
class _NumpyCompatUnpickler(pickle.Unpickler):
    """Loads pickles written under NumPy 2.x on NumPy 1.x (numpy._core is the renamed numpy.core)."""
    def find_class(self, module, name):
        if module == "numpy._core" or module.startswith("numpy._core."):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)

def load_pickle_compat(path):
    with open(path, "rb") as f:
        return _NumpyCompatUnpickler(f).load()

if STAGE in ("all", "select"):
    if "X" not in globals():
        X, feature_names, full_person_index = load_sparse_matrix_from_delta()
        info = check_stored_matrix(X, feature_names)
        FINGERPRINT = matrix_fingerprint(feature_names, X.shape)
    idx_str = pd.Index(full_person_index.astype(str))
    lc = spark.table(CFG["split_table"]).select("person_id", "split", *SPLIT_LABEL_COLS).toPandas()
    lc["person_id"] = lc["person_id"].astype(str)
    assert lc["person_id"].is_unique and len(lc) == len(idx_str), f"split table has {len(lc)} patients, matrix has {len(idx_str)}"
    split_meta = lc.set_index("person_id").reindex(idx_str)
    assert split_meta["split"].notna().all(), "matrix patients missing from the split table"
    split_all = split_meta["split"].to_numpy()
    Y_all = split_meta[SPLIT_LABEL_COLS].to_numpy(dtype=int)

    # (a) the labels this notebook derives from the cohort must equal the labels stored with the split
    coh_pdf = cohort.select(PERSON_ID_COL, *LABEL_COLS).toPandas()
    coh_pdf[PERSON_ID_COL] = coh_pdf[PERSON_ID_COL].astype(str)
    Y_cohort = coh_pdf.set_index(PERSON_ID_COL).reindex(idx_str)[LABEL_COLS].to_numpy(dtype=int)
    assert (Y_cohort == Y_all).all(), f"labels differ from the landmark cohort for {(Y_cohort != Y_all).any(axis=1).sum()} patients"
    # (b) every patient in the original export's test set must be in the split's test set (the same check as 01_landmark_prep). The split may hold EXTRA
    #     test patients: the export lists the patients ProtoPregNet evaluated, and some split patients may have been excluded from that evaluation afterwards.
    if CFG["original_export_path"] and os.path.exists(CFG["original_export_path"]):
        orig_test = set(map(str, load_pickle_compat(CFG["original_export_path"])["pids_test"]))
        new_test = set(idx_str[split_all == "test"])
        missing = orig_test - new_test
        assert not missing, f"{len(missing)}/{len(orig_test)} original test patients are NOT in the split's test set (same check as 01_landmark_prep)"
        extra = new_test - orig_test
        print(f"PASSED: all {len(orig_test):,} original test patients are in the split's test set; the split has {len(extra):,} extra test patients "
              f"({len(extra) / len(new_test):.1%}) that the original export did not evaluate")
    else:
        print("WARNING: original export not found; test-set equality NOT verified")
    print("split sizes:", pd.Series(split_all).value_counts().to_dict(), "| train prevalence (control, pe_ghtn, gdm):",
          Y_all[split_all == "train"].mean(0).round(3).tolist())

In [0]:
# ============================================================================
# stage "select" (2/2): the funnel on TRAIN rows, outputs
# ============================================================================
def add_concept_names(sel):
    """Concept ids / names for the stored columns, via the OMOP concept table. Skipped (with a note) if the table is unavailable."""
    ids = sorted(set(sel["concept_id"].dropna().astype("int64")))
    try:
        c = (spark.table(CFG["concept_table"]).select(F.col("concept_id").cast("long").alias("concept_id"), "concept_name")
             .join(F.broadcast(spark.createDataFrame(pd.DataFrame({"concept_id": ids}))), "concept_id").toPandas())
    except Exception as e:
        print(f"concept names skipped ({type(e).__name__}); set CFG['concept_table'] to enable"); return sel.assign(concept_name=None)
    return sel.merge(c.drop_duplicates("concept_id"), on="concept_id", how="left")

if STAGE in ("all", "select"):
    t0 = time.time()
    tr = np.flatnonzero(split_all == "train")
    Xt, Yt = X[tr], Y_all[tr]
    spec, groups = build_candidate_spec(Xt, info, CFG["min_prevalence"])
    print(f"train rows {len(tr):,} | stored columns {X.shape[1]:,} | groups {len(groups):,} | groups present in >= {CFG['min_prevalence']:.1%} of train patients: "
          f"{int(groups['kept'].sum()):,} -> {len(spec):,} design columns ({int((spec['role'] == 'presence').sum()):,} presence + {int((spec['role'] == 'value').sum()):,} value)")
    print("groups surviving other floors (information only):", {f"{f:.1%}": int((groups["prevalence"] >= f).sum()) for f in CFG["floors_to_report"]})

    res = run_selection(Xt, Yt, spec, CFG, CLASS_NAMES)
    for name in CLASS_NAMES:
        if name in res["cv_tables"]:
            ci = res["cv_info"][name]
            print(f"  L1 {name}: best C {ci['best_C']:.4g} (CV AUROC {ci['best_auroc']:.4f}) -> 1-SE C {ci['chosen_C']:.4g} (CV AUROC {ci['chosen_auroc']:.4f})")

    # ---- the table the landmark notebooks read ----
    sel = spec.merge(info[["col", "concept_id", "window_wk"]], left_on="base_col", right_on="col", how="left").drop(columns="col")
    sel = sel.merge(groups[["group", "prevalence"]], on="group", how="left")
    for stage_key, label in (("fdr", "fdr"), ("screen_group", "screen"), ("funnel", "funnel"), ("final", "final")):
        for name in CLASS_NAMES:
            sel[f"{label}_{name}"] = res[stage_key][name]
    sel["selected_final"] = res["selected_final"]
    sel["matrix_fingerprint"] = FINGERPRINT
    sel = add_concept_names(sel)
    out_cols = ["design_col", "role", "group", "kind", "stat", "base_col", "impute_value", "prevalence", "concept_id", "window_wk", "concept_name",
                *[f"{l}_{n}" for l in ("fdr", "screen", "funnel", "final") for n in CLASS_NAMES], "selected_final", "matrix_fingerprint"]
    sel = sel[out_cols]
    schema = StructType([StructField(c, StringType()) if c in ("design_col", "role", "group", "kind", "stat", "base_col", "concept_name", "matrix_fingerprint")
                         else StructField(c, DoubleType()) if c in ("impute_value", "prevalence")
                         else StructField(c, LongType()) if c in ("concept_id", "window_wk")
                         else StructField(c, BooleanType()) for c in out_cols])
    rows = [tuple(None if (isinstance(v, float) and np.isnan(v)) or v is pd.NA else (int(v) if c in ("concept_id", "window_wk") else (bool(v) if schema[c].dataType == BooleanType() else v))
                  for c, v in zip(out_cols, r)) for r in sel.itertuples(index=False, name=None)]
    (spark.createDataFrame(rows, schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CFG["selected_table"]))

    # ---- stage counts (counts only) ----
    def n_(mask): return int(np.sum(mask))
    stage_rows = [("stored columns", "all", X.shape[1]), ("groups", "all", len(groups)), (f"groups present in >= {CFG['min_prevalence']:.1%} (train)", "all", n_(groups["kept"])),
                  ("candidate design columns", "all", len(spec))]
    for name in CLASS_NAMES:
        stage_rows += [("FDR on presence + imputed values", name, n_(res["fdr"][name])), ("groups passing the presence screen", name, int(spec.loc[res["screen_group"][name], "group"].nunique())),
                       ("lab windows with FDR-selected results that failed the presence screen", name, res["blocked_lab_groups"][name]),
                       ("event concepts with FDR-selected timing that failed the presence screen (not rescued)", name, res["blocked_event_groups"][name]),
                       ("L1 (1-SE) selected", name, n_(res["funnel"][name])), ("FDR AND funnel", name, n_(res["final"][name]))]
    stage_rows += [("selected_final (union over labels)", "all", n_(res["selected_final"])), ("  of which presence columns", "all", n_(res["selected_final"] & (spec["role"] == "presence").to_numpy())),
                   ("  of which value columns", "all", n_(res["selected_final"] & (spec["role"] == "value").to_numpy()))]
    stage_df = pd.DataFrame(stage_rows, columns=["stage", "label", "n"])
    spark.createDataFrame(stage_df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CFG["stage_counts_table"])
    pd.set_option("display.width", 200); pd.set_option("display.max_rows", 100)
    print("\n" + stage_df.to_string(index=False))
    print(f"\nselected_final: {n_(res['selected_final']):,} design columns from {sel.loc[sel['selected_final'], 'group'].nunique():,} groups; "
          f"by kind: {sel.loc[sel['selected_final']].groupby(['kind', 'role']).size().to_dict()}")
    print(f"funnel took {time.time() - t0:.0f}s -> {CFG['selected_table']}")